# vega-lite analysis and creation plot for HW6

## load dataset

In [3]:
# from vega_datasets import data as datasets
import streamlit as st
import altair as alt
import numpy as np
# from vega_datasets import data
import pandas as pd
source = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
df = pd.read_csv(source)
df.head()

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


## Data Exploration and Cleaning

In [4]:
# Preprocess data
# check nan values
df.isna().sum()
# want to see the numerical and categorical variables
numeric_cols = sorted(df.select_dtypes(include=[np.number]).columns.tolist())
categorical_cols = sorted(df.select_dtypes(exclude=[np.number]).columns.tolist())
all_cols = sorted(df.columns.tolist())
# check out columns
# print(df.columns.tolist())
# print(numeric_cols)
# print(categorical_cols)
# explore the dataset further, explore variables that I may use
df["Year Constructed"].unique()
df["Bldg Status"].unique()
df["Total Floors"].unique()
df["Agency Name"].unique()
df["County"].unique()
df["Usage Description"].unique()
df["County"].unique()
# I don't want 0s
df_clean = df[(df["Year Constructed"] != 0) & (df["Total Floors"] != 0) & (df["Year Acquired"] != 0) & (df["Square Footage"] != 0)]
len(df_clean)

# narrow the focus to UIUC buidlings in Champaign and 
df_ui = df_clean[
    (df_clean["Agency Name"] == "University of Illinois") &
    (df_clean["Usage Description"].str.lower() != "not provided") &
    (df_clean["Year Constructed"] >= 1800)
]
df_ui

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
340,University of Illinois,University of Illinois Urbana-Champaign,1301 South Oak Street,Champaign,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1947,1947,3200,1,1,0,Industrial,Industrial,Not provided
341,University of Illinois,University of Illinois Urbana-Champaign,2301 South Lincoln Avenue,Urbana,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1926,1926,20250,3,3,0,Storage,Storage,Not provided
342,University of Illinois,University of Illinois Urbana-Champaign,2301 South Lincoln Avenue,Urbana,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1926,1926,10923,2,2,0,Storage,Storage,Not provided
343,University of Illinois,University of Illinois Urbana-Champaign,2301 South Lincoln Avenue,Urbana,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1910,1910,8450,2,2,0,Storage,Storage,Not provided
344,University of Illinois,University of Illinois Urbana-Champaign,South Race Street,Urbana,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1949,1949,9600,2,2,0,Storage,Storage,Not provided
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8294,University of Illinois,University of Illinois Urbana-Champaign,S Griffith Drive,Champaign,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1987,1987,3200,1,1,0,Industrial,Industrial,Not provided
8295,University of Illinois,University of Illinois Urbana-Champaign,1910 South Giriffith Drive,Champaign,61820,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Use,1995,1996,3468,1,1,0,Storage,Storage,Not provided
8331,University of Illinois,University of Illinois Urbana-Champaign,1302 West Pennsylvania,Urbana,61801,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Progress,2010,2010,35000,4,3,1,Business,Industrial,Not provided
8459,University of Illinois,University of Illinois Urbana-Champaign,306 North Wright Street,Urbana,61801,Champaign,13,Rodney L. Davis,103,Ammons Carol,...,In Progress,2013,2013,230665,7,6,1,Education,Business,Not provided


## First Plot

In [5]:
# First Plot
df_ui = df_ui.copy()

# interaction selection
click_bins = alt.selection_point(fields=["year_bin_start", "year_bin_end"],empty="all")
bin = 5
# make scatter plot
scatter = (alt.Chart(df_ui)
    .transform_bin(
        as_=["year_bin_start", "year_bin_end"],field="Year Constructed",bin=alt.Bin(step=bin))
    .encode(
        x=alt.X("Year Constructed:Q",
            title="Year Constructed",
            scale=alt.Scale(domain=[1800, 2025]),
            axis=alt.Axis(format="d")
        ),
        y=alt.Y("Total Floors:Q",
            title="Total Floors",
            scale=alt.Scale(domain=[0, int(df_ui["Total Floors"].max()) + 1])
        ),
        size=alt.Size("Square Footage:Q",
            title="Square Footage",
            scale=alt.Scale(range=[30, 400])
        ),
        color=alt.Color("Usage Description:N", title="Usage Description"),
        opacity=alt.condition(click_bins, alt.value(0.95), alt.value(0.25)),
        tooltip=[
            alt.Tooltip("Agency Name:N", title="Agency"),
            alt.Tooltip("Location Name:N", title="Location"),
            alt.Tooltip("City:N"),
            alt.Tooltip("County:N"),
            alt.Tooltip("Usage Description:N", title="Usage"),
            alt.Tooltip("Year Constructed:Q", title="Year"),
            alt.Tooltip("Total Floors:Q", title="Floors"),
            alt.Tooltip("Square Footage:Q", title="Sq Ft"),
        ],
    ).mark_circle(strokeWidth=0.6)
    .properties(
        width=800,
        height=400,
        title="University of Illinois Buildings and their Total Floors (1800 - 2025)")
).interactive()

# make histogram
hist = (
    alt.Chart(df_ui)
    .transform_bin(
        as_=["year_bin_start", "year_bin_end"],
        field="Year Constructed",
        bin=alt.Bin(step=bin)
    ).mark_bar()
    .encode(
        x=alt.X(
            "year_bin_start:Q",
            bin="binned",
            title=f"Year Constructed (bins of {bin} years)"
        ),
        x2="year_bin_end:Q",
        y=alt.Y("count():Q", title="Buildings"),
        color=alt.condition(click_bins, alt.value("steelblue"), alt.value("lightblue")),
        tooltip=[
            alt.Tooltip("year_bin_start:Q", title="From Year"),
            alt.Tooltip("year_bin_end:Q", title="To Year"),
            alt.Tooltip("count():Q", title="Number of Buildings"),
        ],
    ).properties(width=800,height=200,title="Year Constructed by Number of Buildings").add_params(click_bins)
)

# combine scatter plot and histogram
plot1 = ((hist & scatter).configure_title(fontSize=20,anchor="middle").configure_axis(titleFontSize=15,labelFontSize=15))
plot1.save('year_building.json')
print(plot1)


alt.VConcatChart(...)


## Second Plot

In [7]:
import altair as alt

agencies = [
    "Department of Agriculture",
    "Department of Central Management Services",
    "Department of Corrections",
    "Department of Human Services",
    "Department of Juvenile Justice",
    "Department of Military Affairs",
    "Department of Natural Resources",
    "Department of Public Health",
    "Department of Revenue",
    "Department of State Police",
    "Department of Transportation",
    "Department of Veterans' Affairs",
]

df_filter = df_clean[
    (df_clean["Agency Name"].isin(agencies))
    & (df_clean["Year Acquired"] >= 2000)
    & (df_clean["Bldg Status"] == "In Use")
].copy()

df_filter.loc[:, "Agency Name"] = df_filter["Agency Name"].str.replace(
    "Department of ", "", regex=False
)

# Give the param a stable name to avoid "param_4_..." collisions
select_agency = alt.selection_point(
    name="select_agency",
    fields=["Agency Name"],
    on="click",
    clear=True
)

bar = (
    alt.Chart(df_filter)
    .mark_bar()
    .encode(
        x=alt.X("count():Q", title="Number of Buildings"),
        y=alt.Y("Agency Name:N", sort="-x", title="Department"),
        color=alt.condition(
            select_agency,
            alt.Color("Agency Name:N", legend=None),
            alt.value("lightgray"),
        ),
        tooltip=[
            alt.Tooltip("Agency Name:N", title="Department"),
            alt.Tooltip("count():Q", title="# of Buildings"),
        ],
    )
    .add_params(select_agency)
    .properties(
        width=500,
        height=350,
        title="In Use Buildings Acquired After 2000 by Government Departments",
    )
)

# OPTIONAL: filter the line chart to selected agency (works with vega-lite v5)
line = (
    alt.Chart(df_filter)
    .transform_filter(select_agency)   # <-- makes the right chart show only the selected dept
    .mark_line(point=True)
    .encode(
        x=alt.X("Year Acquired:O", title="Year Acquired"),
        y=alt.Y("count():Q", title="Number of Acquired Buildings"),
        color=alt.Color("Agency Name:N", title="Department"),
        tooltip=[
            alt.Tooltip("Year Acquired:O", title="Year"),
            alt.Tooltip("Agency Name:N", title="Department"),
            alt.Tooltip("count():Q", title="# of Buildings"),
        ],
    )
    .properties(
        width=350,
        height=300,
        title="Buildings Acquired Over Time by Selected Department",
    )
)

plot2 = (
    (bar | line)
    .configure_axis(titleFontSize=16, labelFontSize=14)
    .configure_title(fontSize=18, anchor="middle")
)

plot2.save("gov_building.json")
plot2


alt.HConcatChart(...)